In [10]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

orders = pd.read_csv(
    "data/olist_delivery_clean.csv",
    parse_dates=["order_purchase_timestamp"]
)

print(orders.shape)
print(orders.columns.tolist())

(96470, 22)
['order_purchase_timestamp', 'customer_unique_id', 'seller_id', 'customer_state', 'seller_state', 'same_state', 'distance_km', 'estimated_delivery_days', 'shipping_limit_days', 'n_items', 'n_sellers', 'n_products', 'total_price', 'total_freight', 'freight_share', 'product_category', 'product_weight_g', 'product_volume_cm3', 'payment_type', 'payment_installments', 'payment_value', 'is_late']


In [11]:
features = ["same_state", "estimated_delivery_days", "product_weight_g"]

X = orders[features]
y = orders["is_late"]

print(X.dtypes)
print(X.isna().sum())

print(y.value_counts())

same_state                   int64
estimated_delivery_days    float64
product_weight_g           float64
dtype: object
same_state                  0
estimated_delivery_days     0
product_weight_g           16
dtype: int64
is_late
0    88644
1     7826
Name: count, dtype: int64


In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

K = 5

model = make_pipeline(
    SimpleImputer(strategy="median"),  
    StandardScaler(),                   
    KNeighborsClassifier(n_neighbors=K) 
)

In [13]:
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score

def evaluate_splits(X, y, splits):
    rows = []

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        fold_model = clone(model)
        fold_model.fit(X_train, y_train)
        predictions = fold_model.predict(X_valid)
        probabilities = fold_model.predict_proba(X_valid)[:, 1]
        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(y_valid, probabilities),
            "precision": precision_score(y_valid, predictions, zero_division=0),
            "recall": recall_score(y_valid, predictions, zero_division=0),
        })

    return pd.DataFrame(rows)

In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

random_results = evaluate_splits(X, y, kf.split(X))
random_results

,fold,roc_auc,precision,recall
0,1,0.533739,0.176471,0.022901
1,2,0.544053,0.244792,0.030679
2,3,0.554470,0.226316,0.027832
3,4,0.551195,0.177778,0.020860
4,5,0.544781,0.217949,0.020694


In [15]:
metrics = ["roc_auc", "precision", "recall"]

random_results[metrics].agg(["mean", "std"])

,roc_auc,precision,recall
mean,0.545648,0.208661,0.024593
std,0.007964,0.030387,0.004459


In [16]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

group_results = evaluate_splits(
    X,
    y,
    gkf.split(X, y, groups=orders["seller_id"])
)

group_results

,fold,roc_auc,precision,recall
0,1,0.548802,0.174107,0.020249
1,2,0.545052,0.194118,0.025424
2,3,0.546222,0.198925,0.022024
3,4,0.536122,0.205000,0.031226
4,5,0.535470,0.173469,0.021131


In [17]:
from sklearn.model_selection import TimeSeriesSplit

orders_time = orders.sort_values(
    "order_purchase_timestamp",
    kind="stable"
).reset_index(drop=True)
X_time = orders_time[features]
y_time = orders_time["is_late"]

tscv = TimeSeriesSplit(n_splits=5)

time_results = evaluate_splits(
    X_time,
    y_time,
    tscv.split(X_time)
)

time_results

,fold,roc_auc,precision,recall
0,1,0.509488,0.045455,0.001383
1,2,0.515916,0.150000,0.001784
2,3,0.526706,0.171429,0.004338
3,4,0.567089,0.099526,0.021807
4,5,0.560983,0.074074,0.018462


In [18]:
def summarize(metric):
    table = pd.DataFrame(
        {
            "KFold": random_results[metric].to_numpy(),
            "GroupKFold": group_results[metric].to_numpy(),
            "TimeSeriesSplit": time_results[metric].to_numpy(),
        },
        index=["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
    ).T
    mean = table.mean(axis=1)
    std = table.std(axis=1)

    table["mean"] = mean
    table["std"] = std

    return table.round(4)


for metric in ["roc_auc", "precision", "recall"]:
    print(metric)
    display(summarize(metric))

roc_auc


,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
KFold,0.5337,0.5441,0.5545,0.5512,0.5448,0.5456,0.0080
GroupKFold,0.5488,0.5451,0.5462,0.5361,0.5355,0.5423,0.0061
TimeSeriesSplit,0.5095,0.5159,0.5267,0.5671,0.5610,0.5360,0.0264


precision


,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
KFold,0.1765,0.2448,0.2263,0.1778,0.2179,0.2087,0.0304
GroupKFold,0.1741,0.1941,0.1989,0.2050,0.1735,0.1891,0.0145
TimeSeriesSplit,0.0455,0.1500,0.1714,0.0995,0.0741,0.1081,0.0523


recall


,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
KFold,0.0229,0.0307,0.0278,0.0209,0.0207,0.0246,0.0045
GroupKFold,0.0202,0.0254,0.0220,0.0312,0.0211,0.0240,0.0045
TimeSeriesSplit,0.0014,0.0018,0.0043,0.0218,0.0185,0.0096,0.0098


В работе использовал KNN с числом соседей `k = 5` и тремя признаками из M0: `same_state`, `estimated_delivery_days` и `product_weight_g`

В весе товара было 16 пропусков. Я заполнил их медианой, потому что на неё меньше влияют очень большие значения, чем на среднее. Также применил StandardScaler: вес измеряется в граммах, срок — в днях, и без масштабирования величина чисел сильно влияет на расстояния в KNN. Медиана и параметры масштабирования вычислялись только на обучающей части каждого фолда. К проверочной части применялись готовые значения, чтобы её данные не влияли на обучение.

Я сравнил KFold, GroupKFold и TimeSeriesSplit, в каждой схеме использовал пять фолдов. Для GroupKFold выбрал группировку по продавцу, чтобы проверить модель на заказах продавцов, которых не было в обучении. Для TimeSeriesSplit отсортировал заказы по дате покупки: обучение происходило на прошлом, а проверка — на следующем участке данных. Обучающая часть постепенно расширялась. Пять проверок позволили оценить результат на нескольких последовательных участках истории.

Средний ROC-AUC составил 0.5456 для KFold, 0.5423 для GroupKFold и 0.5360 для TimeSeriesSplit. Стандартное отклонение между фолдами — 0.0080, 0.0061 и 0.0264.
Для основной оценки и отчёта компании я бы выбрал TimeSeriesSplit, потому что она наиболее естественно воспроизводит будущее использование модели: обучение на прошлых заказах и предсказание следующих

Пока модель получилась слабой... Во временной проверке средний recall равен примерно 0.96%, а precision — 10.81%. Получается, модель пропускает почти все реальные опоздания, а большинство её предупреждений неверные Я предполагаю, что одна из причин - признаки, на которых она обучалась